In [1]:
import torch
# torch.autograd.set_detect_anomaly(True)
from torch.utils.data import DataLoader,WeightedRandomSampler
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torchvision.transforms import v2
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
import re
import segmentation_models_pytorch as smp
from pathlib import Path
from tqdm.notebook import tqdm
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import zarr
import matplotlib
from collections import Counter
%load_ext autoreload
%autoreload 2
###IE###
from utils.helpers import (
    read_images,load_model,bootstrap_model,generate_all_masks,
    make_result_dataloaders,make_training_dataloaders,
    draw_fast_plots,count_class_pixels,effective_num_weights,
    left_mapping_function,TP_TN_FP_FN,get_collapse
)
from utils.preprocessing import (
    normalize_xca,normalize_empty,get_preprocessing,
    remove_black_border_multithread,BrightnessMultiplicativeNNUNet2D,
    ContrastAugmentationNNUNet2D
)
# from utils.debuging import show_bootstraping_results
from models.nnunet import nnUnet
from models.memory_unet import MemoryUnet
from models.swin_encoder import SwinUperNet,SwinUperNet2
from models.swin_upernet_cascade import SwinUperNetCascade
from models.hrnet import HRNET
from models.moe_model import MoeUnet
from models.S3_DSCNet_pro import DSCNet_pro
from models.smp_models import UnetPlPl,DeepLabV3Plus
from models.sarc_unet import SARC_UNet
from utils.testmodel import UTANet
from utils.losses import UnetLoss
from utils.recorder import HistoryRecorder
from scripts.logger import save_full_report
from scripts.trainer import trainer , evaluation
from scripts.image_clusters import valid,problem_maps
from utils.graph_extraction import (
    generate_robust_mask,
    create_gnn_dataset_parallel_limited
)
# from utils.option2_collaps import calc_combined_dice
from models.Swin_Unet.networks.vision_transformer import  SwinUnet
from utils.debuging import weighting_logic , save_loss_weights , save_weight_overlay_cv2
###SS###

/home/parsa/Venvs/AI/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
args = {
    "paths":{
        "base_path" : "./dataset/syntax/",
        "root_path" : "./dataset/syntax/topology_dataset/labels/",
        "output_base_path" : "./outputs",
        "images_folder" : "origianl_preprocessed", # new_preprocessed , origianl_preprocessed , images
        "clahe_folder" : "origianl_preprocessed",
        "generated_folder" : "generated_mask_probs",
        "labels_folder" : "labels",
        "ignore_labels_folder" : "k_fold_preds/2026-08-08_14-07-22/extra_labels",
        "bootstrap_folder" : "bootstrap",
        "hard_vessel_folder" : "hard_vessels_mask",
        "filter_folder" : "filters",
        "bootstrap_seq":"./bootstrap_sequence",
        "topology_folder" : "topology_dataset/labels",
        "side_folder":"./data/",
        "border_cords_folder" : "boarder_removed",
        "skel_folder" : "recall_skel"
        
    },
    "memory_unet":{
        "layer_count":4,
        "channel_max":256,
        "downsampling":"maxpool",
        "dropout" : False,
        "normaliztion":"InstanceNorm2d",#InstanceNorm2d
        "activation" : "LeakyReLU",
        "first_layer_out_c" : 32,
    },
    "swin":{
        "encoder_out_channels":150,
        "warmup_ep" : 0
    },
    "moe_unet" : {                 # 256,   128,    64,    32,     16,     8,      4
        # "encoder_out_channels" : [[128,2],[256,3],[512,5],[1024,2],[1024,3]],
        "encoder_out_channels":[[128,2],[256,3],[512,6],[1024,3],[1024,4]],
        "cbam_args":{"use_cbam":True,"ratio":16,"spatial_kernel":7},
        "dropout_args":{"use":True,"p":0.2},
        "resample_base_size" : None,
        "rp_out_channels" : 64,
        "router_counts":5,
        "expert_counts":3,
        "top_k":2,
        "train_moe":False,
        "norm" : "InstanceNorm2d"#InstanceNorm2d
    },
    "dsc_net":{
        "kernel_size" : 9,
        "extend_scope": 1,
        "if_offset" : True,
        "dim" : 1,
        "numer" : 16
    },
    "socre_params":{
        "p":0.4,
        "r":0.6,
        "choice_score" : "dice"
    },
    "image_shape" : (512,512),
    "in_c":3,
    "deep_super_vision":False,
    "bootstrap":False,
    "training_mode":"multi", # binary - multi - cleaning
    "binary_training_mode" : "partial",# full - partial
    "side_mode":"left", #both - left - right
    "report_class_wise":True,
    "attention" : False,
    "batch_size" : 4,
    "num_workers" : 4,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    "epochs":50,
    "f_int_scale" : 2,
    "new_vessel_width":2,
    "full_report_cycle" : 5,
    "max_channels":512,
    "loss_type":"g_dice",
    "name" : "model[{}]-{}-{}-{}class-{}epo",
    "mask_ce_weights":None,
    "layer_count":5,
    "cut_borders":True,
    "use_ignore_mask" : True,
    ### Loss Functions
    "mask_ce_conf":{
        "imp_coef":1.0
    },
    "mask_frangi_ce_conf": {
        "expert_vessel_pos_weight": 4.0,
        "model_vessel_pos_weight": 3.0,
        "outer_boundary_boost": 4.5,
        "unlabeled_vessel_weight": 0.06,
        "bg_weight": 0.2,
        "kernel_size": 3,
        "imp_coef": 1.0,
        "use_expert_boundary_only": True
    },
    "mask_frangi_dice_conf": {
        "expert_vessel_pos_weight": 4.0,
        "model_vessel_pos_weight": 3.0,
        "outer_boundary_boost": 4.5,
        "unlabeled_vessel_weight": 0.06,
        "kernel_size": 3,
        "bg_weight": 0.3,
        "imp_coef": 1.0,
        "alpha": 0.2,
        "beta": 0.8,
        "use_expert_boundary_only": True
    },
    "ce_loss_conf":{
        "pos_weight":5.0,
        "bg_weight":1.0,
        "small_vessel_weights":10.0,
        "imp_coef":1.0
    },
    "fce_conf":{
        "gamma":2.0,
        "extra_w":3.0,
        "alpha":None
    },
    "tversky_conf":{
        "alpha":0.6,
        "beta":0.4,
        "gamma":1.00,
        "weights":None,
        "imp_coef":1.0
    },
    "g_dice_conf":{
        "imp_coef":1.0
    },
    "aux_loss_conf":{
        "imp_coef" : 0.1
    },
    "cl_dice_conf":{
        "imp_coef":1.0,
        "k":12
    },
    "recall_skel_coef":{
        "imp_coef":1.0
    },
    ### Optimizers
    "sgd_conf":{
        "lr" : 0.1,
        "momentum" : 0.99,
        "weight_decay" : 3e-5,
        "nesterov":True
    },
    "adam_conf":{
        "lr" : 0.01,
        "weight_decay" : 3e-5,
    },
    "adamw_conf":{
        "lr" : 0.0001,
        "weight_decay" : 0.01,
    }
}


if(args["training_mode"] == "binary" or args["training_mode"]=="cleaning"):
    args["class_count"] = 2 
    class_map = {
        1:"fg"
    }
    lut = None
    old_class_map = None,
    collapse = None
else:
    """
        sides : 
            0 - right side
            1 - left side
    """
    if(args["side_mode"] == "left"):
        
        old_class_map = {
            1: '5',2: '6',3: '7',4: '8',
            5: '9',6: '9a',7: '10',8: '10a',
            9: '11',10: '12',11: '12a',12: '13',
            13: '14',14: '14a',15: '15',
            16: '12b',17: '14b'
        }
        collapse , class_map =  get_collapse(option=0)
        lut = left_mapping_function(
            old_class_map = old_class_map,
            new_class_map = class_map,
            collapse = collapse
        )
        args["class_count"] = len(class_map) + 1
        
    elif(args["side_mode"] == "right"):
        # args["class_count"] = 9
        # class_map = {
        #     1: '1',2: '2', 3: '3',4: '4',
        #     5:'16',6: '16a',7: '16b',8: '16c'
        # }
        args["class_count"] = 6
        class_map = {
            1: '1',2: '2', 3: '3',4: '4',
            5:'16'
        }
        lut = None
    else:
        args["class_count"] = 26
        class_map = {
            1: '1',2: '2', 3: '3',4: '4',
            5: '5',6: '6',7: '7',8: '8',
            9: '9',10: '9a',11: '10',12: '10a',
            13: '11',14: '12',15: '12a',16: '13',
            17: '14',18: '14a',19: '15',20: '16',
            21: '16a',22: '16b',23: '16c',
            24: '12b',25: '14b'
        }
        lut = None


print("class_count : ",args["class_count"])


left_hard_classes = [10,11,14,15,17,18,19,24]#9a,10,12,12a,14,14a,15,12b
hard_classes = [20,21,22,23] + left_hard_classes
# losses_keys = ["total loss","FCE loss",args["loss_type"]]
losses_keys = ["total loss","mask CE-Loss",args["loss_type"],"recall skel"]#,"cl_dice_loss","mask CE-Loss" ,"branch_mask CE-Loss" ,args["loss_type"] ,"recall skel"
if(args["deep_super_vision"]):
    shape = args["image_shape"]
    # loss_weights = [1/(2**i) for i in range(1,len(args["moe_unet"]["encoder_out_channels"]))]
    loss_weights = [1/(2**i) for i in range(1,3)]
else:
    loss_weights = []
loss_weights



class_count :  4


[]

In [3]:
train_images,sampler_weights,last_boot_level = read_images(
    paths = args["paths"],
    part = "train",
    chosen_labels=None,
    hard_classes = hard_classes,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map

)
valid_images = read_images(
    paths = args["paths"],
    part = "val",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map
)
test_images = read_images(
    paths = args["paths"],
    part = "test",
    chosen_labels=None,
    bootstrap = args["bootstrap"],
    training_mode = args["training_mode"],
    binary_mode = args["binary_training_mode"],
    side_mode = args["side_mode"],
    in_c = args["in_c"],
    cut_borders = args["cut_borders"],
    class_count = args["class_count"],
    # lut = lut,
    collapse=collapse,
    old_class_map=old_class_map
)

0


  0%|          | 0/1000 [00:00<?, ?it/s]

0


  0%|          | 0/200 [00:00<?, ?it/s]

0


  0%|          | 0/300 [00:00<?, ?it/s]

In [4]:
co = {}

for data in train_images:
    mask = data["label"]
    u = np.unique(mask)
    u= u[1:]
    new_ = Counter(u.tolist())
    co = dict(Counter(co) + Counter(new_))

print(co)

{1: 525, 9: 319, 10: 61, 11: 129, 12: 304, 2: 536, 3: 340, 4: 310, 13: 107, 15: 38, 5: 198, 6: 70, 7: 21, 17: 125, 16: 63, 14: 49, 8: 1}


In [ ]:
class_counts = np.zeros(args["class_count"])
for data in train_images:
    label = data["label"]
    u_idx = np.unique(label)
    class_counts[u_idx]+=1
class_counts
#array([375., 374., 374., 369., 303., 231.,  43.,  48.,  31.])
# array([625., 525., 536., 340., 310., 234.,  22., 319., 211., 304., 205.,38.])

IndexError: index 9 is out of bounds for axis 0 with size 4

In [ ]:
all_class_counts = {}
all_images = {"train":train_images,"valid":valid_images,"test":test_images}
for part , images in all_images.items():
    all_class_counts[part] = {
        "pixel_co":{val:0 for key,val in class_map.items()},
        "class_co":{val:0 for key,val in class_map.items()}
    }
    for data in tqdm(images):
        label = data["label"]
        pixel_counts = Counter(label.reshape(-1).tolist())
        for u in np.unique(label)[1:]:
            val = class_map[u]
            all_class_counts[part]["pixel_co"][val] += pixel_counts[u]
            all_class_counts[part]["class_co"][val] += 1
all_class_counts

  0%|          | 0/625 [00:00<?, ?it/s]

  0%|          | 0/134 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

{'train': {'pixel_co': {'5': 591829,
   '6': 816901,
   '7': 685677,
   '8': 570436,
   '9': 470633,
   '9a': 124025,
   '10': 23866,
   '10a': 1079,
   '11': 507754,
   '12': 151219,
   '12a': 336857,
   '13': 597880,
   '14': 241117,
   '14a': 98167,
   '15': 66890,
   '12b': 164558,
   '14b': 153542},
  'class_co': {'5': 525,
   '6': 536,
   '7': 340,
   '8': 310,
   '9': 198,
   '9a': 70,
   '10': 21,
   '10a': 1,
   '11': 319,
   '12': 61,
   '12a': 129,
   '13': 304,
   '14': 107,
   '14a': 49,
   '15': 38,
   '12b': 63,
   '14b': 125}},
 'valid': {'pixel_co': {'5': 162627,
   '6': 177207,
   '7': 148045,
   '8': 125779,
   '9': 93872,
   '9a': 36464,
   '10': 18708,
   '10a': 3787,
   '11': 100295,
   '12': 35577,
   '12a': 72930,
   '13': 110102,
   '14': 41565,
   '14a': 41069,
   '15': 3715,
   '12b': 26910,
   '14b': 68429},
  'class_co': {'5': 130,
   '6': 125,
   '7': 77,
   '8': 74,
   '9': 50,
   '9a': 25,
   '10': 21,
   '10a': 6,
   '11': 70,
   '12': 17,
   '12a': 27,

In [ ]:
if(args["training_mode"]=="full"):
    b=0.999

    train_class_counts = [
        1200,374,375,369,303,525,537,
        198,340,70,21,310,1,61,320,129,
        63,305,107,49,127,38,232,43,48,31
    ]

    total = np.sum(train_class_counts)
    mask_ce_weights = np.log(total/np.array(train_class_counts))
    mask_ce_weights = (mask_ce_weights / mask_ce_weights.mean()).tolist()
    mask_ce_weights[0]=0.1

    args["mask_ce_weights"]=mask_ce_weights
    args["mask_ce_weights"]
elif(args["training_mode"]=="cleaning"):
    args["mask_ce_weights"] = [0.1,1]
else:
    # class_counts = np.zeros(args["class_count"])
    # for img,hard_idx,label,skel_mask,side,name_stem in train_images:
    #     u_idx = np.unique(label)
    #     class_counts[u_idx]+=1
    # alpha = (1.0 / class_counts)
    # alpha = alpha / alpha.sum() * len(class_counts)
    # args["fce_conf"]["alpha"] = alpha.tolist()

    # class_counts = np.zeros(args["class_count"])
    # for img,hard_idx,label,skel_mask,side,name_stem in train_images:
    #     u_idx = np.unique(label)
    #     counts = np.bincount(label.reshape(-1))
    #     for u in u_idx :
    #         class_counts[u]+=counts[u]
    # alpha = (1.0 / class_counts)
    # alpha = alpha / alpha.sum() * len(class_counts)
    # alpha[0]=np.min(alpha[1:])*0.5
    # args["fce_conf"]["alpha"] = alpha.tolist()
    # args["tversky_conf"]["weights"] = alpha[1:].tolist()

    class_counts = count_class_pixels(train_images=train_images,class_count=args["class_count"])
    alpha = effective_num_weights(class_counts,knee=20,beta=None)
    args["fce_conf"]["alpha"] = alpha.tolist()
    print(f"weights are set to : {alpha}")
    # pass

In [ ]:
train_transforms , test_transforms = get_preprocessing(args["training_mode"],args)

if(args["in_c"]==1):
    if(args["training_mode"]!="cleaning"):
        normalizer = normalize_xca
        print("set to z score")
    else:
        normalizer = normalize_empty
elif(args["in_c"]==2):
    print("set to z score 2c")
    normalizer = normalize_empty
else:
    normalizer = A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    )
    print("set to imagenet")


In [ ]:
# black_th=100
# min_vote=0.85
# remove_black_border_multithread(train_images,paths=args["paths"],part="train",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")
# remove_black_border_multithread(valid_images,paths=args["paths"],part="val",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")
# remove_black_border_multithread(test_images,paths=args["paths"],part="test",
#                                 black_th=black_th,min_vote=min_vote,mode="plot")

In [ ]:
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sampler_weights, dtype=torch.double), 
    num_samples=len(sampler_weights),
    replacement=True
)
train_loader , valid_loader , test_loader = make_training_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    train_transforms=train_transforms,
    test_transforms=test_transforms,
    args=args,
    sampler=sampler
)
re_train_loader , re_valid_loader , re_test_loader  = make_result_dataloaders(
    train_images=train_images,
    valid_images=valid_images,
    test_images=test_images,
    normalizer=normalizer,
    transforms=test_transforms
)

In [ ]:
"""
[0.         3.18589744 3.38461538 2.17948718 1.96794872 1.26923077
 2.03846154 1.95512821]

[0.         3.24358974 3.16666667 1.61538462 1.48717949 0.91666667
 2.6474359  2.57692308]
"""

counts = np.zeros(args["class_count"])
num_batches = 0
for data in tqdm(train_loader):
    label = data["mask"].numpy()
    per_image = []
    for l in label:
        us = np.unique(l)
        per_image.extend(us[us != 0].tolist())
    vals, cnts = np.unique(per_image, return_counts=True) 
    counts[vals.astype(int)] += cnts
    num_batches += 1
avg = counts / num_batches
print(avg)

# Make GNN Dataset

In [ ]:
# generate_robust_mask(
#     model = best_model,
#     train_loader = re_train_loader,
#     valid_loader = re_valid_loader,
#     output_path = "./dataset/syntax/fixed_binary_masks",
#     low_thresh = 0.2,
#     high_thresh = 0.7,
#     save_mode = "plot"
# )

In [ ]:
# import segmentation_models_pytorch as smp


# class Temp(torch.nn.Module):
#     def __init__(self,args):
#         super(Temp,self).__init__()
#         self.model = smp.UnetPlusPlus(
#             encoder_name="resnet50",        # resnet50, efficientnet-b0, timm-*, etc.
#             encoder_weights="imagenet",     # or None for random init
#             in_channels=3,                  # 1 for grayscale/medical
#             classes=9,                      # 1 for binary, N for multiclass
#         )
#     def forward(self,img,s=None):
#         return [self.model(img)]
    
# paths = [
#     "./outputs/2026-07-16_17-29-02 [multi-right-6class-50epo-aux_branchpoint-nobias-cbam-upsample]",
#     "./outputs/2026-07-17_00-28-41 [multi-right-6class-50epo-aux_branchpoint-nobias-cbam-upsample]",
#     "./outputs/2026-07-17_01-04-15 [multi-right-6class-50epo-aux_branchpoint-nobias-cbam-upsample]",
#     "./outputs/2026-07-17_10-41-50 [multi-right-6class-50epo-aux_branchpoint-nobias-cbam-upsample]"
# ]
# class EnsembledModels(torch.nn.Module):
#     def __init__(self, paths):
#         super().__init__()
#         self.models = torch.nn.ModuleList([load_model(model_class= SwinUperNetCascade, path=p,device="cuda",model_inputs={}) for p in paths])
#         for m in self.models:
#             m.eval()

#     @torch.no_grad()
#     def forward(self, img, side=None):
#         probs = torch.stack([
#             F.softmax(m(img, side)[0].float(), dim=1) for m in self.models
#         ]).mean(0)
#         # return LOG-probs: argmax is unchanged, and a downstream softmax is a
#         # no-op that recovers the averaged probs. Drop-in for your eval code.
#         return [torch.log(probs.clamp_min(1e-8))]
# best_model = EnsembledModels(paths=paths)
# best_model.eval()

In [ ]:
# """ 
# 0	bg	            0.999181	0.999137	0.999224	0.998457	0.999264	0.997651	0.998042	0.999352	0.996735
# 1	16	            0.913923	0.925558	0.902577	0.877687	0.852899	0.903958	0.840416	0.798548	0.886918
# 2	16a	            0.840075	0.827115	0.853448	0.465802	0.310353	0.933242	0.391509	0.250586	0.894622
# 3	16b	            0.866188	0.859638	0.872839	0.796416	0.707907	0.910221	0.668194	0.524188	0.921294
# 4	16c	            0.837810	0.832076	0.843623	0.777782	0.685738	0.898366	0.640092	0.496513	0.900491
# 5	binary score	0.896530	0.901483	0.891630	0.826378	0.757548	0.908965	0.749786	0.642233	0.900610
# 6	mean score	    0.864499	0.861097	0.868122	0.729422	0.639224	0.911447	0.635053	0.517459	0.900831

# missed 16 branch in img 101
# missed 16 branch in img 103
# missed 16 branch in img 11
# missed 16 branch in img 179
# """

# model_name = "2026-07-15_14-42-51 [multi-right-6class-150epo-aux_branchpoint-nobias-cbam-upsample]"
# out_folder = f"./outputs/{model_name}"
# # best_model,args,train_transforms = load_model(
# #     model_class=MoeUnet,
# #     path=out_folder,
# #     model_inputs={},
# #     device="cuda",
# #     return_args = True
# # )
# # best_model = torch.load(f"{out_folder}/model.pkl",weights_only=False)
# # best_model.eval()

# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader
# }
# base_path = f"./dataset/syntax/gnn_dataset/{model_name}"

# create_gnn_dataset_parallel_limited(
#     model=best_model,
#     dataloaders=dataloaders,
#     args=args,
#     base_path=base_path,
#     low_thresh=0.2,
#     high_thresh=0.7,
#     enhance_mask=False,
#     paths=args["paths"],
#     labeling_mode="binary_kernel",
#     r=1, #2
#     min_votes=1, #15
#     save_pred_masks=True,
#     max_workers=4,
#     max_pending=8,
#     class_map = {0:"bg",1:"16",2:"16a",3:"16b",4:"16c"}
# )

# Bootstrap

In [ ]:
# out_folders = [
#     "./outputs/2026-05-31_19-37-00 [binary-tests-dsv-frangi+swin-boot_skel]",
# ]
# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader,
# }
# last_boot_level = 3
# for out_folder in out_folders:
    
#     best_model = load_model(
#         model_class=SwinUperNet,
#         path=out_folder,
#         model_inputs={},
#         device=args["device"]
#     )
#     bootstrap_model(
#         model = best_model,
#         last_boot_level = last_boot_level,
#         args = args,
#         dataloaders = dataloaders,
#         gt_mask_mode = "pred_skel", #real_pred , pred_skel 
#         model_name = out_folder.split("/")[2],
#         iterations = 2,
#         new_vessel_width = args["new_vessel_width"],
#         comp = False,
#         use_erode = False
#     )
#     last_boot_level+=1
    

# Trained Model Outputs

In [ ]:
# out_folder = "./outputs/2026-04-16_19-46-59 [binary-tests-dsv-clahe+sobel+swin]++/"
# best_model = load_model(
#     model_class=SwinUperNet,
#     path=out_folder,
#     model_inputs={"args":args},
#     device=args["device"]
# )

# df = model_performance_per_image(
#     model=best_model,
#     dataloader=re_valid_loader,
#     args=args,
#     out_folder="./data/model_performance",
#     valid_path="./data/valid_problem_df.csv"
# )
# generate_all_masks(
#     model=best_model,
#     train_loader=re_train_loader,
#     valid_loader=re_valid_loader,
#     test_loader=re_test_loader,
#     output_base="./dataset/syntax/"
# )

# generate_robust_mask(
#     model=best_model,
#     valid_loader=re_valid_loader,
#     train_loader=re_train_loader,
#     output_path="./dataset/syntax/fixed_binary_masks"
# )

In [ ]:
# names=[]
# co = np.zeros(9)
# for data in train_loader: 
#     name = data["name"]
#     label = data["mask"]
#     for i in range(label.shape[0]):
#         l = label[i]
#         u = np.unique(l)
#         co[u]+=1
#     names+=name
# Counter(names)
# co

# model = SwinUperNet(args)
# from torchview import draw_graph

# g = draw_graph(model, input_size=(1, 3, 512, 512),graph_dir="LR")
# g.visual_graph  

In [ ]:
def load_binary_backbone(multi_model, ckpt_path, device="cpu", verbose=True):
    """
    Initialise SwinUperNet2 (multiclass) from a trained SwinUperNet (binary).
    Transfers ONLY `self.model.*` -- the full HF UperNet (Swin encoder + UperNet decoder).
    """
    sd = torch.load(ckpt_path, map_location=device)
    sd = sd.get("state_dict", sd.get("model", sd))
    sd = { (k[7:] if k.startswith("module.") else k): v for k, v in sd.items() }

    backbone = {k: v for k, v in sd.items() if k.startswith("model.")}
    dropped  = sorted({k.split('.')[0] for k in sd if not k.startswith("model.")})

    missing, unexpected = multi_model.load_state_dict(backbone, strict=False)
    bad = [k for k in missing if k.startswith("model.")]

    if verbose:
        print(f"transferred {len(backbone)} backbone tensors")
        print(f"dropped from binary ckpt   : {dropped}")
        print(f"randomly init in multiclass: "
              f"{sorted({k.split('.')[0] for k in missing if not k.startswith('model.')})}")
        print(f"backbone keys NOT loaded   : {len(bad)}  <-- must be 0")
    assert not bad, f"backbone mismatch: {bad[:5]}"
    return multi_model

In [ ]:
class Temp(torch.nn.Module):
    def __init__(self, args):
        super().__init__()
        self.model = load_model(
            model_class=SwinUperNet,
            path="./outputs/2026-08-11_02-30-49 [model[SwinUperNet]-multi-left-7class-50epo]",
            model_inputs={},
            device="cuda",
        )
        for p in self.model.parameters():      
            p.requires_grad = False
        self.model.eval()       

        self.second_model = SwinUperNet(args)

        self.fuse = torch.nn.Conv2d(3 + 7, 3, kernel_size=1)

    def train(self, mode=True):
        super().train(mode)
        self.model.eval()           
        return self

    def forward(self, x, gt_mask=None, ep=None):
        with torch.no_grad():
            big_out, _ = self.model(x)     
            big_logits = big_out[0].detach()
            big_probs = F.softmax(big_logits,dim=1)
        fused = self.fuse(torch.cat([x, big_probs], dim=1))
        return self.second_model(fused)

In [ ]:
# model = load_model(
#     path="./outputs/2026-08-11_15-41-46 [model[Temp]-multi-left-6class-50epo]",
#     model_class = Temp,
#     model_inputs={},
#     device="cuda"
# )

In [ ]:
# import numpy as np
# import torch

# BASE_CLASS_MAP = {1:'5', 2:'12', 3:'6', 4:'9', 5:'7', 6:'9a+10', 7:'8+10a',
#                   8:'11', 9:'12a+12b', 10:'13', 11:'14+14a+14b+15'}

# # stage-1 family idx -> baseline idx for the family's MAIN trunk
# FAMILY_MAIN  = {1:1, 2:3, 3:5, 4:7, 5:8, 6:10}     # 5, 6, 7, 8, 11, 13
# # stage-1 family idx -> baseline idx for that family's CHILDREN
# FAMILY_CHILD = {1:2, 2:4, 3:6, 4:7, 5:9, 6:11}     # 12, 9, 9a+10, 10a, 12a+12b, 14...
# #   family 4 -> 7 for both, because the baseline merges '8+10a' into one class
# # stage-2 main idx -> baseline idx
# MAIN_BASE    = {1:1, 2:3, 3:5, 4:7, 5:8, 6:10}


# def _lut(d, device):
#     a = np.zeros(max(d) + 1, np.int64)
#     for k, v in d.items():
#         a[k] = v
#     return torch.as_tensor(a, device=device)


# def combine(logits_family, logits_main, restrict_main_to_family=True):
#     """
#     logits_family : (B,7,H,W) stage-1, bg + 6 FAMILIES (main + its children)
#     logits_main   : (B,7,H,W) stage-2, bg + 6 MAIN trunks
#     returns       : (B,H,W) int64 labels in the 11-class baseline space

#     Logic: a family pixel is a CHILD unless stage 2 says it is the main trunk.
#     restrict_main_to_family : ignore stage-2 main predictions that fall outside any
#         family stage 1 found. Stage 1 defines where vessels are; a main prediction in
#         empty space is a false positive with no family to belong to.
#     """
#     cf = logits_family.argmax(1)                       # family idx per pixel
#     cm = logits_main.argmax(1)                         # main idx per pixel
#     dev = cf.device

#     fam_child = _lut(FAMILY_CHILD, dev)[cf]
#     main_base = _lut(MAIN_BASE, dev)[cm]

#     has_fam  = cf > 0
#     has_main = cm > 0
#     if restrict_main_to_family:
#         has_main = has_main & has_fam

#     out = torch.zeros_like(cf)
#     out[has_fam] = fam_child[has_fam]                   # family - main = children
#     out[has_main] = main_base[has_main]                 # stage 2 carves out the trunk
#     return out


# def dice_scores(preds, gts, n_classes=12, eps=1e-8):
#     inter = np.zeros(n_classes); psum = np.zeros(n_classes); gsum = np.zeros(n_classes)
#     for pr, gt in zip(preds, gts):
#         pr = np.asarray(pr); gt = np.asarray(gt)
#         for c in range(1, n_classes):
#             p, g = (pr == c), (gt == c)
#             inter[c] += (p & g).sum(); psum[c] += p.sum(); gsum[c] += g.sum()
#     per = {}
#     for c in range(1, n_classes):
#         if psum[c] == 0 and gsum[c] == 0:
#             continue
#         per[BASE_CLASS_MAP[c]] = 2 * inter[c] / (psum[c] + gsum[c] + eps)
#     macro = float(np.mean(list(per.values()))) if per else 0.0
#     return macro, per

In [ ]:


# preds, gts = [], []
# model = load_model(
#     path="./outputs/2026-08-11_21-58-43 [model[Temp]-multi-left-7class-50epo]",
#     model_class = Temp,
#     model_inputs={},
#     device="cuda"
# )
# model.eval()
# device = "cuda"
# with torch.no_grad():
#     for data in tqdm(valid_loader):                 # gt_base already in the 11-class space
#         img = data["imgs"].to(device)
#         gt_base = data["mask"]

#         fam_out, _  = model.model(img)      # stage 1: 7-family (frozen)
#         main_out, _ = model(img)            # Temp -> stage 2: mains only
#         out = combine(fam_out[0], main_out[0])
#         preds += list(out.cpu().numpy())
#         gts   += list(gt_base.numpy())

# macro, per_class = dice_scores(preds, gts)
# print(round(macro, 4))
# print({k: round(v, 3) for k, v in per_class.items()})

# Train The Model

In [ ]:
if(args["training_mode"]!="cleaning"):
    # model = Temp(args=args)
    # model = load_binary_backbone(model, "./outputs/2026-08-01_23-56-16 [model[SwinUperNet]-binary-left-2class-50epo-aux_branchpoint-nobias-cbam-upsample]/model.pth")

    model = SwinUperNet(args=args)
    # model = SwinUperNetCascade(args=args)
    # model = MemoryUnet(args=args)
    # model = MoeUnet(args)
    # model = DSCNet_pro(args)
    # model = DeepLabV3Plus(args)
    # model = UnetPlPl(args)
    # model = Tnet().cuda()
    # model.apply(init_resnet_style)
    # model = UTANet(pretrained=True, n_classes=2)
    
    # model = load_model(
    #     model_class=SwinUperNet,
    #     path="./outputs/2026-07-25_19-17-05 [model[SwinUperNet]-binary-left-2class-50epo-aux_branchpoint-nobias-cbam-upsample]",
    #     model_inputs = {},
    #     device="cuda",   
    # )
    # model.dsv0_conv = torch.nn.Conv2d(512,args["class_count"],kernel_size=1)
    # model.dsv1_conv = torch.nn.Conv2d(512,args["class_count"],kernel_size=1)
    # model.dsv2_conv = torch.nn.Conv2d(512,args["class_count"],kernel_size=1)
    # model.conv1x1 = torch.nn.Conv2d(150,args["class_count"],kernel_size=1)
    # for param in model.model.backbone.parameters():
    #     param.requires_grad = False

    # model.model.backbone.eval()

    # model = model.to(args["device"])
    # optimizer = torch.optim.SGD(
    #     model.parameters(),
    #     **args["sgd_conf"]
    # )

    # for name, p in model.model.named_parameters():
    #     if any(f"encoder.layers.{i}" in name for i in (0, 1, 2)):
    #         p.requires_grad = False
    model = model.to(args["device"])

    # frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    # total  = sum(p.numel() for p in model.parameters())
    # print(f"frozen {frozen/1e6:.1f}M / {total/1e6:.1f}M  ({frozen/total:.1%})")
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        **args["adamw_conf"]
    )
   
    # backbone_params = list(model.model.parameters())
    # head_params = [p for n, p in model.named_parameters() if not n.startswith("model.")]

    # optimizer = torch.optim.AdamW([
    #     {"params": backbone_params, "lr": args["adamw_conf"]["lr"] * 0.1},
    #     {"params": head_params,     "lr": args["adamw_conf"]["lr"]},
    # ], **{k: v for k, v in args["adamw_conf"].items() if k != "lr"})
    
else:
    model = MemoryUnet(args=args)
    model = model.to(args["device"])

    optimizer = torch.optim.SGD(
        model.parameters(),
        **args["sgd_conf"]
    )

loss_fn = UnetLoss(args)

lr_sch = PolynomialLR(optimizer=optimizer,total_iters=args["epochs"],power=0.9)
# lr_sch = None
recorder = HistoryRecorder(losses_keys=losses_keys,class_maps =class_map,
                           class_count=args["class_count"]-1)


best_model = trainer(
    args=args,
    recorder = recorder,
    model = model,
    optimizer = optimizer,
    loss_fn = loss_fn,
    train_loader = train_loader,
    valid_loader = valid_loader,
    test_loader=test_loader,
    loss_weights=loss_weights,
    lr_sch = lr_sch,
    choice_score= args["socre_params"]["choice_score"]
)
# show_bootstraping_results(args)


# Test Evaluation

In [ ]:
# # model_name = "2026-06-20_17-23-58 [binary-costum+moe+branchpoint_masks-nobias-cbam-upsample]"
# # out_folder = f"./outputs/{model_name}/"
# # best_model = load_model(
# #     model_class=MoeUnet,
# #     path=out_folder,
# #     model_inputs={"args":args},
# #     device=args["device"]
# # )
# import segmentation_models_pytorch as smp


# # paths = [  
# #     ["./outputs/2026-07-28_13-46-04 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],
# #     ["./outputs/2026-07-27_21-19-14 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],
# #     ["./outputs/2026-07-28_15-57-59 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],
# #     ["./outputs/2026-07-28_03-08-21 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],
# #     ["./outputs/2026-07-27_19-09-51 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],

# #     ["./outputs/2026-08-03_03-14-25 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2],
# #     ["./outputs/2026-08-02_23-27-45 [model[SwinUperNet2]-multi-left-7class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet2]
# # ]
# paths = [  
#     ["./outputs/2026-08-07_22-49-38 [model[SwinUperNet]-multi-left-4class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],
#     # ["./outputs/2026-08-08_01-42-47 [model[SwinUperNet]-multi-left-4class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],
#     # ["./outputs/2026-08-08_17-58-08 [model[SwinUperNet]-multi-left-4class-50epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],
#     ["./outputs/2026-08-08_19-50-41 [model[SwinUperNet]-multi-left-4class-100epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],
#     ["./outputs/2026-08-09_00-02-08 [model[SwinUperNet]-multi-left-4class-100epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],

#     # ["./outputs/2026-08-09_02-09-40 [model[SwinUperNet]-multi-left-4class-100epo-aux_branchpoint-nobias-cbam-upsample]",SwinUperNet],
# ]
# class EnsembledModels(torch.nn.Module):
#     def __init__(self, paths):
#         super().__init__()
#         self.models = torch.nn.ModuleList([load_model(model_class= c, path=p,device="cuda",model_inputs={}) for p,c in paths])
#         for m in self.models:
#             m.eval()

#     @torch.no_grad()
#     def forward(self, img, gt_mask=None,ep=None):
#         probs = torch.stack([
#             F.softmax(m(img)[0][0].float(), dim=1) for m in self.models
#         ]).mean(0)
#         # binary_preds = self.binary_model(img)[0][0]
#         # binary_preds = torch.argmax(binary_preds,dim=1)
#         # return LOG-probs: argmax is unchanged, and a downstream softmax is a
#         # no-op that recovers the averaged probs. Drop-in for your eval code.
#         return [torch.log(probs.clamp_min(1e-8))],0
    
# best_model = EnsembledModels(paths=paths)
# best_model.eval()
# dice_score = evaluation(
#     recorder=recorder,
#     model = best_model,
#     loss_fn = loss_fn,
#     valid_loader = valid_loader,
#     class_count = args["class_count"],
#     training_mode = args["training_mode"],
#     class_wise_report = True,
#     epoch = 1,
#     device = "cuda"

# )
# print(dice_score)

# Model Results

In [ ]:
# dataloaders = {
#     "train":re_train_loader,
#     "val":re_valid_loader,
#     "test":re_test_loader,
# }
# input_boot_strap_path = Path("./dataset/syntax/bootstrap")
# output_results_path = Path("./bootstrap_sequence")

# steps = os.listdir(input_boot_strap_path)
# steps = list(map(int, steps))
# steps.sort()

# for part,dataloader in dataloaders.items():
#     out_path = output_results_path/part
#     os.makedirs(out_path,exist_ok=True)

#     for img_base, img, mask, side, name in tqdm(dataloader):
#         image = img_base.squeeze(0).permute(1,2,0).numpy().copy()
#         name = str(name[0])
#         gt_image = image.copy()
#         gt_image[mask.numpy()[0]!=0] = [0,0,255]
        
#         masks = [image,gt_image]
#         for step in steps : 
#             input_path = input_boot_strap_path / str(step) / part / f"{name}.png"
#             b_mask = cv2.imread(input_path,cv2.IMREAD_GRAYSCALE)
#             img = image.copy()
#             img[b_mask!=0] = [0,0,255]
#             masks.append(img)
#         full_plot = draw_fast_plots(masks,plot_shape=(2,3))
#         cv2.imwrite(out_path / f"{name}.png",full_plot)

In [ ]:
args["name"] = args["name"].format(best_model._get_name(),args["training_mode"],args["side_mode"],args["class_count"],args["epochs"])

# Save Results

In [ ]:
save_full_report(
    recorder= recorder , 
    output_base_path=args["paths"]["output_base_path"],
    model=best_model,
    valid_loader=valid_loader,
    normalizer = normalizer,
    args=args,
    class_map=class_map,
    name=args["name"],
    valid_images = valid_images,
    test_transforms = test_transforms,
    training_transforms = train_transforms,
    notebook_name = "Multi_Main.ipynb",
    class_count = args["class_count"],
    device = args["device"],
    training_mode = args["training_mode"],
    side_mode = args["side_mode"],
    last_boot_level=last_boot_level,
    all_class_counts = all_class_counts
)